# Figure 4 — analysis (muscle stem cells: young, aged, geriatric)
Computes every Figure 4 / Supplementary Figure 4Sa / 4Sb result and writes it to `data/muscle/`; the panels are drawn
from these tables in `muscle_figures.ipynb`; the 4Sb page is assembled in `muscle_pages.ipynb` (Figure 4 and 4Sa are
assembled in Illustrator, `../../illustrator-figures/Figure4.ai`, `Figure4Sa.ai`). The NIA replication (Supplementary
Figure 4S NIA) is self-contained in `nia_replication.ipynb`.

Dataset: Zhao et al. 2023, young (2 mo) / aged (23–24 mo) / geriatric (28–32 mo). Interchromosomal LAS contact graphs,
per-age LAS coolers, salmon quantifications and PyDESeq2 results are read from `/data/land/3d-inter-paper-data/`. Runs
in ~15 min in total, most of it the net-accumulation sweep (~10 min).

In [1]:
import itertools
import json
import os
import pickle
import sys
from collections import Counter

import cooler
import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import false_discovery_control, norm

sys.path.insert(0, "..")                                # shared modules
import coloc_stats                                      # contact counting, per-state hypergeometric test + BH
import gsea                                             # preranked GSEA on the DESeq2 Wald statistic
import kcore                                            # state-specific k-cores
import locus_null                                       # locus-matched permutation test

META = "../../metadata"
ZHAO = "/data/land/3d-inter-paper-data/muscle-sc-zhao"
LAS = f"{ZHAO}/runs/pooled-v2/analysis"                 # Zhao G_all graphs
ZHAO_RNA = f"{ZHAO}/raw-data/rna"                       # Zhao PyDESeq2 results
ZHAO_QUANT = f"{ZHAO}/raw-data/rna/quant"                # salmon quantification per sample
COOL = ZHAO + "/runs/pooled-v2/cool/{age}_INTER_LAS_250000_filt_{filt}.cool"   # per-age interchromosomal LAS contacts
ANNOT = "/data/land/3d-inter-paper-data/cd8-hu/annot"
MM10_BINS = f"{ANNOT}/mm10_250kb/mm10_250kb_bins_annotated.csv"   # one row per 250 kb bin; row = graph node / cooler bin id
TX2GENE = f"{ANNOT}/gencode.vM25.tx2gene.tsv"
OUT = "data/muscle"
FILT = "20.0"
SIG = 0.05
PATHWAYS = json.load(open(f"{META}/mm10_hallmark_and_custom_pathways.json"))   # 60 Hallmark + custom (mouse)
# (group A, group B) per Zhao contrast: A fills the "young/earlier" slot, B the "old/later" slot
CONTRASTS = [("young", "aged"), ("young", "geriatric"), ("aged", "geriatric")]
AGES = ["young", "aged", "geriatric"]
bins = pd.read_csv(MM10_BINS)
print(f"{len(PATHWAYS)} gene sets, {len(bins)} bins")

60 gene sets, 9859 bins


## Pathway colocalization (Figure 4b, 4c, 4e; Supplementary Figure 4Sa, panel b)
For each pathway and contrast (shared `../coloc_stats.py`): unique gene–gene contacts per state, per-state
hypergeometric test against the
genome-wide background (BH over pathways), the Fisher test of the state skew (saved for reference), and the fold
enrichment of each state's contacts relative to its genome-wide share. Writes
`data/muscle/pathway_enrichment_<pair>_filt_20.0.csv`, `differential_specificity_<pair>_filt_20.0.csv` and
`fold_enrichment_<pair>.csv`.

In [2]:
def enrichment_table(G, A, B):
    """Per pathway: state-specific contact counts, shares, hypergeometric p and BH FDR, and an enrichment label."""
    states = [A, B]
    bg = coloc_stats.count_contacts(G, "protein_coding_genes_with_tss")
    M = sum(bg.get(s, 0) for s in states) + bg.get("Shared", 0)
    rows = []
    for name, genes in PATHWAYS.items():
        row = coloc_stats.count_contacts(G, "protein_coding_genes_with_tss", genes)
        N = sum(row.get(s, 0) for s in states) + row.get("Shared", 0)
        rec = {"pathway": name, "n_pathway_contacts": N}
        for s in states:
            k = row.get(s, 0)
            rec[f"n_{s}"] = k
            rec[f"pct_{s}"] = 100 * k / N if N else np.nan
            rec[f"pval_{s}"] = coloc_stats.hypergeom_enrichment_pval(k, M, bg.get(s, 0), N) if N else np.nan
        rows.append(rec)
    res = pd.DataFrame(rows)
    for s in states:
        res[f"fdr_{s}"] = coloc_stats.bh(res[f"pval_{s}"])
        res[f"nlog10_fdr_{s}"] = -np.log10(res[f"fdr_{s}"])

    def label(r):
        a = r[f"fdr_{A}"] < 0.05
        b = r[f"fdr_{B}"] < 0.05
        if a and b:
            return "Both"
        if a:
            return f"{A} only"
        if b:
            return f"{B} only"
        return "None"

    res["enrichment"] = res.apply(label, axis=1)
    return res


def specificity_table(bg, A, B, enr):
    """Fisher exact on [[n_B, n_A], [bg_B, bg_A]] per pathway: does it lean B vs A more than the genome? OR > 1 = B."""
    res = enr.copy()
    odds_ratios = []
    pvals = []
    for _, r in res.iterrows():
        odds_ratio, p = coloc_stats.differential_fisher(r[f"n_{B}"], r[f"n_{A}"], bg[B], bg[A])
        odds_ratios.append(odds_ratio)
        pvals.append(p)
    res["fisher_or"] = odds_ratios
    res["fisher_p"] = pvals
    res["fisher_fdr"] = coloc_stats.bh(res["fisher_p"])
    return res


def fold_table(enr, bg, A, B):
    """Fold enrichment per state = pathway share / genome-wide share; signed_effect > 0 = B (older) skewed."""
    total = bg.get(A, 0) + bg.get(B, 0) + bg.get("Shared", 0)
    pct_a = 100 * bg.get(A, 0) / total
    pct_b = 100 * bg.get(B, 0) / total
    e = enr.set_index("pathway")
    out = pd.DataFrame(index=e.index)
    out["fold_young"] = e[f"pct_{A}"] / pct_a             # "young" = the earlier group A, "old" = B
    out["fold_old"] = e[f"pct_{B}"] / pct_b
    out["fdr_young"] = e[f"fdr_{A}"]
    out["fdr_old"] = e[f"fdr_{B}"]
    out["sig_young"] = out["fdr_young"] < SIG
    out["sig_old"] = out["fdr_old"] < SIG
    out["signed_effect"] = (out["fold_old"] - 1) - (out["fold_young"] - 1)
    return out

In [3]:
for A, B in CONTRASTS:
    pair = f"{A}_vs_{B}"
    G = pickle.load(open(f"{LAS}/G_all_{pair}_filt_{FILT}.pkl", "rb"))
    bg = coloc_stats.count_contacts(G, "protein_coding_genes_with_tss")
    res = enrichment_table(G, A, B)
    res.sort_values(f"nlog10_fdr_{A}", ascending=False).to_csv(f"{OUT}/pathway_enrichment_{pair}_filt_{FILT}.csv",
                                                               index=False)
    enr = pd.read_csv(f"{OUT}/pathway_enrichment_{pair}_filt_{FILT}.csv")          # as saved (sorted)
    spec = specificity_table(bg, A, B, enr)
    spec.sort_values("fisher_p").to_csv(f"{OUT}/differential_specificity_{pair}_filt_{FILT}.csv", index=False)
    fold_table(enr, bg, A, B).to_csv(f"{OUT}/fold_enrichment_{pair}.csv")
    print(f"{pair}: {(res.enrichment != 'None').sum()} enriched; Fisher FDR < 0.05: {int((spec.fisher_fdr < 0.05).sum())}")

young_vs_aged: 21 enriched; Fisher FDR < 0.05: 25


young_vs_geriatric: 15 enriched; Fisher FDR < 0.05: 25


aged_vs_geriatric: 11 enriched; Fisher FDR < 0.05: 7


## Pathway expression (GSEA) (Supplementary Figure 4Sa, panels a, c, d; Figure 4c)
Preranked GSEA per contrast on the PyDESeq2 Wald statistic (shared `../gsea.py`). Writes `data/muscle/gsea_<pair>.csv`.

In [4]:
for A, B in CONTRASTS:
    de = pd.read_csv(f"{ZHAO_RNA}/de_{A}_vs_{B}.csv", index_col=0)
    res = gsea.prerank(gsea.deseq_rank(de), PATHWAYS)
    res.to_csv(f"{OUT}/gsea_{A}_vs_{B}.csv", index=False)
    print(f"{A}_vs_{B}: {(res['FDR q-val'] < 0.05).sum()} sets at FDR < 0.05")

2026-10-21 01:44:06,249 [WARNING] Duplicated values found in preranked stats: 28.54% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


2026-10-21 01:44:15,500 [WARNING] Duplicated values found in preranked stats: 34.62% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


young_vs_aged: 52 sets at FDR < 0.05


2026-10-21 01:44:25,937 [WARNING] Duplicated values found in preranked stats: 31.95% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


young_vs_geriatric: 47 sets at FDR < 0.05


aged_vs_geriatric: 45 sets at FDR < 0.05


## Locus-matched permutation test (Supplementary Figure 4Sb, panels a, b)
Each pathway's fold enrichment of state-specific contacts against random gene-bearing loci on the same chromosome
matched for gene density and expression, 10,000 draws, shared `../locus_null.py` (seed = 1000 + pathway position).
Expression per age = salmon read counts summed per gene (gencode vM25), averaged over the replicates of each age; it only
sets coarse expression strata. Writes `data/muscle/expression_by_age.csv` and `data/muscle/locus_matched_<pair>.csv`.

In [5]:
SAMPLES = {"SRR17072254": "young", "SRR17072253": "young",
           "SRR17072252": "aged", "SRR17072251": "aged", "SRR17072250": "aged",
           "SRR17072249": "geriatric", "SRR17072248": "geriatric", "SRR17072247": "geriatric"}
tx2gene = pd.read_csv(TX2GENE, sep="\t")
tx_to_gene = dict(zip(tx2gene.iloc[:, 0].str.split(".").str[0], tx2gene.iloc[:, 1]))   # transcript (no version) -> symbol

sample_counts = {}
for srr in SAMPLES:
    quant = pd.read_csv(f"{ZHAO_QUANT}/{srr}/quant.sf", sep="\t", usecols=["Name", "NumReads"])
    transcript = quant.Name.str.split("|").str[0].str.split(".").str[0]
    sample_counts[srr] = quant.NumReads.groupby(transcript.map(tx_to_gene)).sum()
counts = pd.DataFrame(sample_counts).fillna(0)
expr = counts.T.groupby(pd.Series(SAMPLES)).mean().T                    # gene x age
expr.to_csv(f"{OUT}/expression_by_age.csv")
print(expr.shape, "genes with counts:", (expr.sum(axis=1) > 0).sum())

(54242, 3) genes with counts: 35741


In [6]:
for pair, a, b in [("young_vs_geriatric", "young", "geriatric"), ("aged_vs_geriatric", "aged", "geriatric")]:
    G = pickle.load(open(f"{LAS}/G_all_{pair}_filt_{FILT}.pkl", "rb"))
    expr_by_state = {s: expr[s] for s in (a, b)}
    locus_null.prepare(G, bins, (a, b), expr_by_state, gene_col="protein_coding_genes_with_tss")
    locus = locus_null.test_family(PATHWAYS)
    locus.to_csv(f"{OUT}/locus_matched_{pair}.csv", index=False)
    print(pair, "p < 0.05:", {s: int((locus[f"locus_p_{s}"] < 0.05).sum()) for s in (a, b)})

young_vs_geriatric p < 0.05: {'young': 1, 'geriatric': 4}


aged_vs_geriatric p < 0.05: {'aged': 3, 'geriatric': 4}


## State-specific k-cores (Figure 4d; Supplementary Figure 4Sb, panel c)
Densest k-core of the geriatric-only colocalization graph (young vs geriatric) among a pathway's genes (shared
`../kcore.py`; each gene on its first TSS bin), and the gene × gene contact-state matrix. Writes
`data/muscle/kcores/<core>_{genes,states}.csv` and `kcore_summary.csv`.

In [7]:
# a gene with TSSs in several bins is placed on its first bin, so each heatmap row is one locus (the colocalization
# tests above count every TSS bin of a gene)
gene_bin = {}                                           # gene -> first 250 kb bin holding its TSS
for i, value in bins["protein_coding_genes_with_tss"].items():
    if isinstance(value, str) and value:
        for g in value.split(","):
            gene_bin.setdefault(g, i)
gene_bins = {g: {b} for g, b in gene_bin.items()}
bin_chrom = {i: int(c.removeprefix("chr")) for i, c in bins["chrom"].items()}

G_geri = pickle.load(open(f"{LAS}/G_all_young_vs_geriatric_filt_{FILT}.pkl", "rb"))
edge_type = nx.get_edge_attributes(G_geri, "edge_type")
STATE_CODE = {"young": 1, "Shared": 2, "geriatric": 3}  # heatmap codes; 0 = no contact
CORES = [("ifng", "Interferon gamma response (H)"), ("ifna", "Interferon alpha response (H)"), ("complement", "Complement (H)")]

os.makedirs(f"{OUT}/kcores", exist_ok=True)
summary = []
for name, pathway in CORES:
    genes = [g for g in PATHWAYS[pathway] if g in gene_bin]
    module, k = kcore.state_kcore(edge_type, gene_bins, genes, "geriatric", bin_chrom)
    mat = kcore.state_matrix(edge_type, gene_bins, module, STATE_CODE)
    chroms = [bins.loc[gene_bin[g], "chrom"] for g in module]
    pd.DataFrame({"gene": module, "chrom": chroms}).to_csv(f"{OUT}/kcores/{name}_genes.csv", index=False)
    pd.DataFrame(mat, index=module, columns=module).to_csv(f"{OUT}/kcores/{name}_states.csv")
    counts = {s: int((mat == c).sum() // 2) for s, c in STATE_CODE.items()}
    summary.append({"core": name, "pathway": pathway, "k": k, "n_genes": len(module), "n_chromosomes": len(set(chroms)),
                    **{f"pairs_{s}": n for s, n in counts.items()}})
    print(f"{pathway}: {k}-core, {len(module)} genes on {len(set(chroms))} chromosomes | {counts}")
pd.DataFrame(summary).to_csv(f"{OUT}/kcores/kcore_summary.csv", index=False)

Interferon gamma response (H): 13-core, 37 genes on 8 chromosomes | {'young': 9, 'Shared': 112, 'geriatric': 323}
Interferon alpha response (H): 8-core, 30 genes on 8 chromosomes | {'young': 4, 'Shared': 80, 'geriatric': 179}
Complement (H): 8-core, 49 genes on 13 chromosomes | {'young': 31, 'Shared': 371, 'geriatric': 291}


## Age trajectory of colocalization (Figure 4f, 4g; Supplementary Figure 4Sa, panel e)
Counted at the gene–gene level from the per-age interchromosomal LAS coolers (a contact between two 250 kb bins counts
for every pair of genes with a TSS in them). Terms:
- **LAS budget** of an age: the number of genome-wide interchromosomal gene pairs in contact at that age.
- **Fold enrichment** of a pathway at an age: the fraction of its interchromosomal gene pairs in contact / the
  genome-wide fraction (LAS budget / all possible interchromosomal gene pairs).
- **Excess over null** (4f, 4Sa e): fold / the median fold of 300 random gene sets of one size (176 genes, the median
  mapped size of the four old-enriched pathways); 4f divides by the young value.
- **Repertoire** of a pathway: its interchromosomal gene pairs in contact at any age. 4f and 4Sa e show the highlighted
  pathways plus every other pathway with a repertoire of at least 200 pairs (smaller ones are noisy).
- **Highlighted pathways**: old-enriched = interferon-γ, interferon-α, complement, fibrosis; young-enriched = bile acid
  metabolism, UV response down, KRAS signaling down.
- **Net accumulation** (4g): (pairs gained after young and kept, i.e. absent in young and present in geriatric, − pairs
  present only in young) / repertoire; pairs seen only transiently are excluded. Tested against 500 random gene sets of
  the pathway's own size (Gaussian z, two-sided; BH over the 60 pathways).

Random draws use `np.random.seed(0)`, set before the 300-draw null and again before the sweep. Writes
`data/muscle/trajectory_{relative,absolute}.csv` and `data/muscle/net_accumulation_sweep.csv`.

In [8]:
# bin -> genes with a TSS in it, and gene -> its bins (row index = cooler bin id)
chrom = np.asarray(bins["chrom"].astype(str))
bin_genes = [[] for _ in range(len(bins))]
gene_to_bins = {}
for bin_id, gene_str in enumerate(bins["protein_coding_genes_with_tss"].astype("object")):
    if isinstance(gene_str, str) and gene_str:
        bin_genes[bin_id] = gene_str.split(",")
        for gene in bin_genes[bin_id]:
            gene_to_bins.setdefault(gene, set()).add(bin_id)
all_genes = np.array(list(gene_to_bins), dtype=object)

In [9]:
# per age: interchromosomal LAS contacts (bin pairs) and the LAS budget (gene pairs in contact genome-wide)
las_edges = {}
gene_budget = {}
for age in AGES:
    pixels = cooler.Cooler(COOL.format(age=age, filt=FILT)).pixels()[:]
    las_edges[age] = set(zip(pixels["bin1_id"].values, pixels["bin2_id"].values))
    seen = set()
    for b1, b2 in las_edges[age]:
        for g1 in bin_genes[b1]:
            for g2 in bin_genes[b2]:
                if g1 != g2:
                    seen.add((g1, g2) if g1 < g2 else (g2, g1))
    gene_budget[age] = len(seen)

In [10]:
# possible trans gene-gene pairs genome-wide (denominator of the background rate; fixed across ages)
gene_chrom = {gene: chrom[min(bin_set)] for gene, bin_set in gene_to_bins.items()}
per_chrom_genes = Counter(gene_chrom.values())
within_chrom_gene_pairs = sum(count * (count - 1) // 2 for count in per_chrom_genes.values())
GENOME_GENE_PAIRS = len(gene_chrom) * (len(gene_chrom) - 1) // 2 - within_chrom_gene_pairs
print("gene-gene LAS budget:", gene_budget, f"| {GENOME_GENE_PAIRS:,} possible trans gene pairs")

gene-gene LAS budget: {'young': 13799838, 'aged': 16522918, 'geriatric': 13901525} | 201,821,902 possible trans gene pairs


In [11]:
def pathway_gene_pairs(genes):
    """All trans gene-gene pairs among a pathway's loci, each mapped to its carrying bin pair."""
    genes_in_bin = {}
    for gene in set(genes):
        for bin_id in gene_to_bins.get(gene, ()):
            genes_in_bin.setdefault(bin_id, []).append(gene)
    pairs = {}
    for b1, b2 in itertools.combinations(sorted(genes_in_bin), 2):
        if chrom[b1] == chrom[b2]:
            continue
        bin_pair = (min(b1, b2), max(b1, b2))
        for g1 in genes_in_bin[b1]:
            for g2 in genes_in_bin[b2]:
                if g1 == g2:
                    continue
                pairs.setdefault((g1, g2) if g1 < g2 else (g2, g1), bin_pair)
    return pairs


def fold_vector(genes):
    """Fold enrichment (young, aged, geriatric): pathway gene-pair LAS density / genome-wide rate; None if unmappable."""
    pairs = pathway_gene_pairs(genes)
    if len(pairs) == 0:
        return None
    folds = []
    for age in AGES:
        n_contact = sum(bp in las_edges[age] for bp in pairs.values())
        folds.append(n_contact / len(pairs) / (gene_budget[age] / GENOME_GENE_PAIRS))
    return np.array(folds)


def pattern_counts(genes):
    """Count a pathway's trans gene pairs by presence pattern across (young, aged, geriatric); also the repertoire."""
    counts = {}
    for bin_pair in pathway_gene_pairs(genes).values():
        pattern = tuple(int(bin_pair in las_edges[age]) for age in AGES)
        if pattern != (0, 0, 0):
            counts[pattern] = counts.get(pattern, 0) + 1
    return counts, sum(counts.values())


def net_accumulation(genes):
    """(gained-and-kept - young-lost) / repertoire; gained-kept = (0,1,1) + (0,0,1), young-lost = (1,0,0).
    Transient classes (aged-only, young+aged, young+geriatric) are excluded."""
    counts, repertoire = pattern_counts(genes)
    if repertoire == 0:
        return np.nan
    gained_kept = counts.get((0, 1, 1), 0) + counts.get((0, 0, 1), 0)
    young_lost = counts.get((1, 0, 0), 0)
    return (gained_kept - young_lost) / repertoire


def net_stats(genes, n_draw=500):
    """Net accumulation vs a size-matched random-gene null (Gaussian z, two-sided p)."""
    _, repertoire = pattern_counts(genes)
    observed = net_accumulation(genes)
    size = len([g for g in genes if g in gene_to_bins])
    null = np.array([net_accumulation(np.random.choice(all_genes, size=size, replace=False)) for _ in range(n_draw)])
    mu = float(np.nanmean(null))
    sd = float(np.nanstd(null, ddof=1))
    z = (observed - mu) / sd if sd > 0 else 0.0
    return observed, mu, z, float(2 * norm.sf(abs(z))), repertoire

In [12]:
# random-gene null: 300 gene sets at the median mapped size of the old-enriched pathways
np.random.seed(0)
OLD_ENRICHED = ["Interferon gamma response (H)", "Interferon alpha response (H)", "Complement (H)", "Fibrosis (O)"]
YOUNG_ENRICHED = ["Bile acid metabolism (H)", "Uv response dn (H)", "KRAS signaling dn (H)"]
null_size = int(np.median([len([g for g in PATHWAYS[n] if g in gene_to_bins]) for n in OLD_ENRICHED]))
null = []
for _ in range(300):
    vec = fold_vector(np.random.choice(all_genes, size=null_size, replace=False))
    if vec is not None:
        null.append(vec)
null_p50 = np.percentile(np.array(null), 50, axis=0)
print(f"null gene-set size {null_size}; random-gene median fold {null_p50.round(3)}")

null gene-set size 176; random-gene median fold [0.989 0.992 0.982]


In [13]:
# excess over the null per age: highlighted pathways and every other pathway with repertoire >= 200
MIN_REP = 200
highlight = set(OLD_ENRICHED) | set(YOUNG_ENRICHED)
levels = {}
for name in PATHWAYS:
    _, repertoire = pattern_counts(PATHWAYS[name])
    if name not in highlight and repertoire < MIN_REP:
        continue
    vec = fold_vector(PATHWAYS[name])
    if vec is not None:
        levels[name] = vec / null_p50                   # absolute excess over null
relative = {name: lev / lev[0] for name, lev in levels.items()}   # relative to young (young = 1)
pd.DataFrame(levels, index=AGES).T.to_csv(f"{OUT}/trajectory_absolute.csv", index_label="pathway")
pd.DataFrame(relative, index=AGES).T.to_csv(f"{OUT}/trajectory_relative.csv", index_label="pathway")
print(len(levels), "pathways in the trajectory panels")

54 pathways in the trajectory panels


In [14]:
# net-accumulation sweep over the 60 pathways; heavy, ~10 min
np.random.seed(0)
sweep_rows = []
for name in PATHWAYS:
    net, null_mean, z, p, repertoire = net_stats(PATHWAYS[name])
    sweep_rows.append(dict(pathway=name, repertoire=repertoire, net=net, null_mean=null_mean,
                           effect=net - null_mean, z=z, p=p))
sweep = pd.DataFrame(sweep_rows)
sweep["fdr"] = false_discovery_control(sweep["p"].values, method="bh")
sweep["direction"] = np.where(sweep["z"] > 0, "gained", "lost")
sweep = sweep.sort_values("effect", ascending=False).reset_index(drop=True)
sweep.to_csv(f"{OUT}/net_accumulation_sweep.csv", index=False)
print(f"{len(sweep)} pathways | FDR < 0.05: {(sweep.fdr < 0.05).sum()} "
      f"(gained {((sweep.fdr < 0.05) & (sweep.direction == 'gained')).sum()}, "
      f"lost {((sweep.fdr < 0.05) & (sweep.direction == 'lost')).sum()})")

60 pathways | FDR < 0.05: 4 (gained 3, lost 1)
